# Crop-Specific Field Condition Assessment Engine

## Project Objective

The purpose of this notebook is to develop the first decision-support component
of the Smart Agriculture AI Platform.

The farmer will first select the crop being cultivated:

- Rice
- Sugarcane

The system will then analyze field conditions obtained from ESP32-based IoT
sensors and assess whether the current conditions are suitable for the selected
crop.

The system will evaluate:

- Soil Moisture
- Temperature
- Humidity
- Rainfall
- Soil pH
- Nitrogen (N)
- Phosphorus (P)
- Potassium (K)

The output will provide:

1. Individual parameter status
2. Overall field condition
3. Identification of problematic parameters
4. Actionable recommendations

This module does not recommend which crop the farmer should grow.

The farmer selects the crop first, and the system evaluates the current field
conditions specifically for that selected crop.

This rule-based assessment engine will serve as a baseline decision-support
layer before integrating supervised machine learning models for irrigation,
fertilizer, and soil health recommendations.

In [1]:
import pandas as pd
import numpy as np

print("Libraries imported successfully.")

Libraries imported successfully.


## Sensor Input Schema

The production system is expected to receive the following environmental and
soil measurements from the ESP32 IoT hardware.

These values will eventually be received through the API layer.

For the current notebook, sensor values will be provided manually to simulate
ESP32 input.

In [3]:
SENSOR_FEATURES = [
    "soil_moisture",
    "temperature",
    "humidity",
    "rainfall",
    "ph",
    "nitrogen",
    "phosphorus",
    "potassium"
]

print("Expected ESP32 sensor features:")
for feature in SENSOR_FEATURES:
    print("-", feature)

Expected ESP32 sensor features:
- soil_moisture
- temperature
- humidity
- rainfall
- ph
- nitrogen
- phosphorus
- potassium


In [4]:
SUPPORTED_CROPS = [
    "rice",
    "sugarcane"
]

print("Supported crops:", SUPPORTED_CROPS)

Supported crops: ['rice', 'sugarcane']


In [5]:
CROP_REQUIREMENTS = {

    "rice": {

        "temperature": {
            "optimal": (20, 30),
            "acceptable": (18, 35)
        },

        "humidity": {
            "optimal": (60, 80),
            "acceptable": (50, 90)
        },

        "soil_moisture": {
            "optimal": (60, 80),
            "acceptable": (40, 90)
        },

        "rainfall": {
            "optimal": (150, 300),
            "acceptable": (100, 400)
        },

        "ph": {
            "optimal": (5.5, 7.0),
            "acceptable": (5.0, 7.5)
        },

        "nitrogen": {
            "optimal": (60, 120),
            "acceptable": (40, 140)
        },

        "phosphorus": {
            "optimal": (30, 70),
            "acceptable": (20, 80)
        },

        "potassium": {
            "optimal": (40, 80),
            "acceptable": (30, 100)
        }
    },


    "sugarcane": {

        "temperature": {
            "optimal": (20, 35),
            "acceptable": (18, 40)
        },

        "humidity": {
            "optimal": (60, 80),
            "acceptable": (50, 90)
        },

        "soil_moisture": {
            "optimal": (60, 80),
            "acceptable": (40, 90)
        },

        "rainfall": {
            "optimal": (150, 300),
            "acceptable": (100, 400)
        },

        "ph": {
            "optimal": (6.0, 7.5),
            "acceptable": (5.5, 8.0)
        },

        "nitrogen": {
            "optimal": (80, 150),
            "acceptable": (50, 180)
        },

        "phosphorus": {
            "optimal": (30, 70),
            "acceptable": (20, 90)
        },

        "potassium": {
            "optimal": (80, 150),
            "acceptable": (50, 180)
        }
    }
}

In [6]:
def validate_sensor_data(sensor_data):

    required_features = set(SENSOR_FEATURES)

    provided_features = set(sensor_data.keys())

    missing_features = required_features - provided_features

    if missing_features:
        raise ValueError(
            f"Missing sensor features: {missing_features}"
        )

    for feature in SENSOR_FEATURES:

        value = sensor_data[feature]

        if value is None:
            raise ValueError(
                f"{feature} cannot be None."
            )

        if not isinstance(value, (int, float)):
            raise TypeError(
                f"{feature} must be numeric."
            )

        if not np.isfinite(value):
            raise ValueError(
                f"{feature} must be a finite number."
            )

    return True

In [7]:
def assess_parameter(value, optimal_range, acceptable_range):

    optimal_min, optimal_max = optimal_range
    acceptable_min, acceptable_max = acceptable_range

    if optimal_min <= value <= optimal_max:

        return "OPTIMAL"

    elif acceptable_min <= value <= acceptable_max:

        return "WARNING"

    else:

        return "CRITICAL"

In [8]:
def assess_crop_condition(crop, sensor_data):

    crop = crop.lower().strip()

    # Validate crop
    if crop not in SUPPORTED_CROPS:
        raise ValueError(
            f"Unsupported crop: {crop}. "
            f"Supported crops are: {SUPPORTED_CROPS}"
        )

    # Validate sensor data
    validate_sensor_data(sensor_data)

    requirements = CROP_REQUIREMENTS[crop]

    results = {}

    for parameter in SENSOR_FEATURES:

        value = sensor_data[parameter]

        optimal_range = requirements[parameter]["optimal"]

        acceptable_range = requirements[parameter]["acceptable"]

        status = assess_parameter(
            value,
            optimal_range,
            acceptable_range
        )

        results[parameter] = {
            "value": value,
            "status": status,
            "optimal_range": optimal_range,
            "acceptable_range": acceptable_range
        }

    # Count statuses

    optimal_count = sum(
        1 for result in results.values()
        if result["status"] == "OPTIMAL"
    )

    warning_count = sum(
        1 for result in results.values()
        if result["status"] == "WARNING"
    )

    critical_count = sum(
        1 for result in results.values()
        if result["status"] == "CRITICAL"
    )

    # Overall condition

    if critical_count > 0:

        overall_condition = "CRITICAL"

    elif warning_count >= 2:

        overall_condition = "WARNING"

    else:

        overall_condition = "OPTIMAL"

    return {
        "crop": crop,
        "parameters": results,
        "summary": {
            "optimal": optimal_count,
            "warning": warning_count,
            "critical": critical_count
        },
        "overall_condition": overall_condition
    }

In [9]:
rice_sensor_data = {

    "soil_moisture": 70,

    "temperature": 27,

    "humidity": 72,

    "rainfall": 220,

    "ph": 6.5,

    "nitrogen": 80,

    "phosphorus": 45,

    "potassium": 60
}

In [10]:
rice_result = assess_crop_condition(
    crop="rice",
    sensor_data=rice_sensor_data
)

rice_result

{'crop': 'rice',
 'parameters': {'soil_moisture': {'value': 70,
   'status': 'OPTIMAL',
   'optimal_range': (60, 80),
   'acceptable_range': (40, 90)},
  'temperature': {'value': 27,
   'status': 'OPTIMAL',
   'optimal_range': (20, 30),
   'acceptable_range': (18, 35)},
  'humidity': {'value': 72,
   'status': 'OPTIMAL',
   'optimal_range': (60, 80),
   'acceptable_range': (50, 90)},
  'rainfall': {'value': 220,
   'status': 'OPTIMAL',
   'optimal_range': (150, 300),
   'acceptable_range': (100, 400)},
  'ph': {'value': 6.5,
   'status': 'OPTIMAL',
   'optimal_range': (5.5, 7.0),
   'acceptable_range': (5.0, 7.5)},
  'nitrogen': {'value': 80,
   'status': 'OPTIMAL',
   'optimal_range': (60, 120),
   'acceptable_range': (40, 140)},
  'phosphorus': {'value': 45,
   'status': 'OPTIMAL',
   'optimal_range': (30, 70),
   'acceptable_range': (20, 80)},
  'potassium': {'value': 60,
   'status': 'OPTIMAL',
   'optimal_range': (40, 80),
   'acceptable_range': (30, 100)}},
 'summary': {'optimal'

In [11]:
def display_assessment(result):

    print("=" * 60)

    print(
        f"Crop: {result['crop'].upper()}"
    )

    print(
        f"Overall Condition: "
        f"{result['overall_condition']}"
    )

    print("=" * 60)

    for parameter, details in result["parameters"].items():

        print(
            f"{parameter:20} | "
            f"Value: {details['value']:8} | "
            f"Status: {details['status']}"
        )

    print("=" * 60)

    print("Summary")

    print(
        "Optimal:",
        result["summary"]["optimal"]
    )

    print(
        "Warning:",
        result["summary"]["warning"]
    )

    print(
        "Critical:",
        result["summary"]["critical"]
    )

In [12]:
display_assessment(rice_result)

Crop: RICE
Overall Condition: OPTIMAL
soil_moisture        | Value:       70 | Status: OPTIMAL
temperature          | Value:       27 | Status: OPTIMAL
humidity             | Value:       72 | Status: OPTIMAL
rainfall             | Value:      220 | Status: OPTIMAL
ph                   | Value:      6.5 | Status: OPTIMAL
nitrogen             | Value:       80 | Status: OPTIMAL
phosphorus           | Value:       45 | Status: OPTIMAL
potassium            | Value:       60 | Status: OPTIMAL
Summary
Optimal: 8
Critical: 0


In [13]:
rice_problem_sensor_data = {

    "soil_moisture": 20,

    "temperature": 38,

    "humidity": 45,

    "rainfall": 50,

    "ph": 8.0,

    "nitrogen": 20,

    "phosphorus": 15,

    "potassium": 20
}

In [14]:
rice_problem_result = assess_crop_condition(
    crop="rice",
    sensor_data=rice_problem_sensor_data
)

display_assessment(rice_problem_result)

Crop: RICE
Overall Condition: CRITICAL
soil_moisture        | Value:       20 | Status: CRITICAL
temperature          | Value:       38 | Status: CRITICAL
humidity             | Value:       45 | Status: CRITICAL
rainfall             | Value:       50 | Status: CRITICAL
ph                   | Value:      8.0 | Status: CRITICAL
nitrogen             | Value:       20 | Status: CRITICAL
phosphorus           | Value:       15 | Status: CRITICAL
potassium            | Value:       20 | Status: CRITICAL
Summary
Optimal: 0
Critical: 8


In [15]:
sugarcane_sensor_data = {

    "soil_moisture": 70,

    "temperature": 29,

    "humidity": 70,

    "rainfall": 220,

    "ph": 6.8,

    "nitrogen": 110,

    "phosphorus": 50,

    "potassium": 110
}

In [16]:
sugarcane_result = assess_crop_condition(
    crop="sugarcane",
    sensor_data=sugarcane_sensor_data
)

display_assessment(sugarcane_result)

Crop: SUGARCANE
Overall Condition: OPTIMAL
soil_moisture        | Value:       70 | Status: OPTIMAL
temperature          | Value:       29 | Status: OPTIMAL
humidity             | Value:       70 | Status: OPTIMAL
rainfall             | Value:      220 | Status: OPTIMAL
ph                   | Value:      6.8 | Status: OPTIMAL
nitrogen             | Value:      110 | Status: OPTIMAL
phosphorus           | Value:       50 | Status: OPTIMAL
potassium            | Value:      110 | Status: OPTIMAL
Summary
Optimal: 8
Critical: 0


In [17]:
selected_crop = "rice"

current_sensor_data = rice_sensor_data

result = assess_crop_condition(
    crop=selected_crop,
    sensor_data=current_sensor_data
)

display_assessment(result)

Crop: RICE
Overall Condition: OPTIMAL
soil_moisture        | Value:       70 | Status: OPTIMAL
temperature          | Value:       27 | Status: OPTIMAL
humidity             | Value:       72 | Status: OPTIMAL
rainfall             | Value:      220 | Status: OPTIMAL
ph                   | Value:      6.5 | Status: OPTIMAL
nitrogen             | Value:       80 | Status: OPTIMAL
phosphorus           | Value:       45 | Status: OPTIMAL
potassium            | Value:       60 | Status: OPTIMAL
Summary
Optimal: 8
Critical: 0


In [18]:
selected_crop = "sugarcane"

current_sensor_data = sugarcane_sensor_data

result = assess_crop_condition(
    crop=selected_crop,
    sensor_data=current_sensor_data
)

display_assessment(result)

Crop: SUGARCANE
Overall Condition: OPTIMAL
soil_moisture        | Value:       70 | Status: OPTIMAL
temperature          | Value:       29 | Status: OPTIMAL
humidity             | Value:       70 | Status: OPTIMAL
rainfall             | Value:      220 | Status: OPTIMAL
ph                   | Value:      6.8 | Status: OPTIMAL
nitrogen             | Value:      110 | Status: OPTIMAL
phosphorus           | Value:       50 | Status: OPTIMAL
potassium            | Value:      110 | Status: OPTIMAL
Summary
Optimal: 8
Critical: 0


# Actionable Recommendation Engine

The Crop Condition Assessment Engine identifies whether each field parameter
is optimal, warning, or critical for the selected crop.

The next step is to convert these diagnostic results into actionable,
farmer-friendly recommendations.

The recommendation engine will:

1. Identify problematic parameters.
2. Generate a recommendation for each parameter.
3. Prioritize critical issues.
4. Provide a final summary for the farmer.

The farmer remains responsible for selecting the crop.
The system does not recommend a different crop.

In [19]:
def generate_parameter_recommendation(
    parameter,
    value,
    status,
    crop
):

    if status == "OPTIMAL":

        recommendations = {

            "soil_moisture":
                "Soil moisture is within the optimal range. No immediate irrigation action is required.",

            "temperature":
                "Temperature is within the suitable range for the selected crop.",

            "humidity":
                "Humidity is within the suitable range for the selected crop.",

            "rainfall":
                "Rainfall conditions are within the expected range.",

            "ph":
                "Soil pH is within the suitable range. No immediate pH correction is indicated.",

            "nitrogen":
                "Nitrogen level is within the target range. No immediate nitrogen intervention is indicated.",

            "phosphorus":
                "Phosphorus level is within the target range. No immediate phosphorus intervention is indicated.",

            "potassium":
                "Potassium level is within the target range. No immediate potassium intervention is indicated."
        }

        return recommendations.get(
            parameter,
            "Parameter is within the optimal range."
        )


    elif status == "WARNING":

        recommendations = {

            "soil_moisture":
                "Soil moisture is outside the optimal range. Monitor field moisture closely and evaluate irrigation requirements.",

            "temperature":
                "Temperature is outside the optimal range. Monitor the crop for possible temperature stress.",

            "humidity":
                "Humidity is outside the optimal range. Monitor environmental conditions and crop health.",

            "rainfall":
                "Rainfall is outside the optimal range. Monitor water availability and adjust irrigation planning if necessary.",

            "ph":
                "Soil pH is outside the optimal range. Consider soil testing and appropriate soil management.",

            "nitrogen":
                "Nitrogen is outside the optimal range. Consider soil testing and nutrient management based on crop stage.",

            "phosphorus":
                "Phosphorus is outside the optimal range. Consider soil testing and nutrient management.",

            "potassium":
                "Potassium is outside the optimal range. Consider soil testing and nutrient management."
        }

        return recommendations.get(
            parameter,
            "Monitor this parameter closely."
        )


    else:

        recommendations = {

            "soil_moisture":
                "Soil moisture is outside the acceptable range. Immediate field inspection is recommended. Evaluate irrigation requirements before taking action.",

            "temperature":
                "Temperature is outside the acceptable range. The crop may be experiencing environmental stress. Monitor conditions closely.",

            "humidity":
                "Humidity is outside the acceptable range. Monitor crop conditions and potential environmental stress.",

            "rainfall":
                "Rainfall is outside the acceptable range. Review water availability and irrigation requirements.",

            "ph":
                "Soil pH is outside the acceptable range. Conduct a soil test and consult an agronomist before applying corrective inputs.",

            "nitrogen":
                "Nitrogen is outside the acceptable range. Conduct soil testing before applying fertilizer.",

            "phosphorus":
                "Phosphorus is outside the acceptable range. Conduct soil testing before applying fertilizer.",

            "potassium":
                "Potassium is outside the acceptable range. Conduct soil testing before applying fertilizer."
        }

        return recommendations.get(
            parameter,
            "Immediate investigation of this parameter is recommended."
        )

In [20]:
def generate_crop_recommendations(assessment_result):

    crop = assessment_result["crop"]

    recommendations = []

    for parameter, details in assessment_result["parameters"].items():

        recommendation = generate_parameter_recommendation(
            parameter=parameter,
            value=details["value"],
            status=details["status"],
            crop=crop
        )

        recommendations.append({

            "parameter": parameter,

            "value": details["value"],

            "status": details["status"],

            "recommendation": recommendation
        })


    # Identify critical parameters
    critical_parameters = [

        item["parameter"]

        for item in recommendations

        if item["status"] == "CRITICAL"
    ]


    # Identify warning parameters
    warning_parameters = [

        item["parameter"]

        for item in recommendations

        if item["status"] == "WARNING"
    ]


    # Overall action

    if critical_parameters:

        overall_action = (
            "Immediate attention is required for critical parameters. "
            "Verify sensor readings and inspect field conditions."
        )

    elif warning_parameters:

        overall_action = (
            "Some field conditions are outside the optimal range. "
            "Monitor the field and take appropriate corrective action."
        )

    else:

        overall_action = (
            "Current field conditions are within the defined optimal ranges. "
            "Continue regular monitoring."
        )


    return {

        "crop": crop,

        "overall_condition":
            assessment_result["overall_condition"],

        "recommendations":
            recommendations,

        "critical_parameters":
            critical_parameters,

        "warning_parameters":
            warning_parameters,

        "overall_action":
            overall_action
    }

In [21]:
rice_recommendations = generate_crop_recommendations(
    rice_result
)

rice_recommendations

{'crop': 'rice',
 'overall_condition': 'OPTIMAL',
 'recommendations': [{'parameter': 'soil_moisture',
   'value': 70,
   'status': 'OPTIMAL',
   'recommendation': 'Soil moisture is within the optimal range. No immediate irrigation action is required.'},
  {'parameter': 'temperature',
   'value': 27,
   'status': 'OPTIMAL',
   'recommendation': 'Temperature is within the suitable range for the selected crop.'},
  {'parameter': 'humidity',
   'value': 72,
   'status': 'OPTIMAL',
   'recommendation': 'Humidity is within the suitable range for the selected crop.'},
  {'parameter': 'rainfall',
   'value': 220,
   'status': 'OPTIMAL',
   'recommendation': 'Rainfall conditions are within the expected range.'},
  {'parameter': 'ph',
   'value': 6.5,
   'status': 'OPTIMAL',
   'recommendation': 'Soil pH is within the suitable range. No immediate pH correction is indicated.'},
  {'parameter': 'nitrogen',
   'value': 80,
   'status': 'OPTIMAL',
   'recommendation': 'Nitrogen level is within the t

In [22]:
def display_recommendations(result):

    print("=" * 75)

    print(
        f"Crop: {result['crop'].upper()}"
    )

    print(
        f"Overall Condition: "
        f"{result['overall_condition']}"
    )

    print("=" * 75)


    for item in result["recommendations"]:

        print(
            f"\nParameter: {item['parameter']}"
        )

        print(
            f"Value: {item['value']}"
        )

        print(
            f"Status: {item['status']}"
        )

        print(
            f"Recommendation: "
            f"{item['recommendation']}"
        )


    print("\n" + "=" * 75)

    print(
        "FINAL ACTION:"
    )

    print(
        result["overall_action"]
    )

    print("=" * 75)

In [23]:
display_recommendations(
    rice_recommendations
)

Crop: RICE
Overall Condition: OPTIMAL

Parameter: soil_moisture
Value: 70
Status: OPTIMAL
Recommendation: Soil moisture is within the optimal range. No immediate irrigation action is required.

Parameter: temperature
Value: 27
Status: OPTIMAL
Recommendation: Temperature is within the suitable range for the selected crop.

Parameter: humidity
Value: 72
Status: OPTIMAL
Recommendation: Humidity is within the suitable range for the selected crop.

Parameter: rainfall
Value: 220
Status: OPTIMAL
Recommendation: Rainfall conditions are within the expected range.

Parameter: ph
Value: 6.5
Status: OPTIMAL
Recommendation: Soil pH is within the suitable range. No immediate pH correction is indicated.

Parameter: nitrogen
Value: 80
Status: OPTIMAL
Recommendation: Nitrogen level is within the target range. No immediate nitrogen intervention is indicated.

Parameter: phosphorus
Value: 45
Status: OPTIMAL
Recommendation: Phosphorus level is within the target range. No immediate phosphorus intervention

In [24]:
rice_problem_recommendations = generate_crop_recommendations(
    rice_problem_result
)

display_recommendations(
    rice_problem_recommendations
)

Crop: RICE
Overall Condition: CRITICAL

Parameter: soil_moisture
Value: 20
Status: CRITICAL
Recommendation: Soil moisture is outside the acceptable range. Immediate field inspection is recommended. Evaluate irrigation requirements before taking action.

Parameter: temperature
Value: 38
Status: CRITICAL
Recommendation: Temperature is outside the acceptable range. The crop may be experiencing environmental stress. Monitor conditions closely.

Parameter: humidity
Value: 45
Status: CRITICAL
Recommendation: Humidity is outside the acceptable range. Monitor crop conditions and potential environmental stress.

Parameter: rainfall
Value: 50
Status: CRITICAL
Recommendation: Rainfall is outside the acceptable range. Review water availability and irrigation requirements.

Parameter: ph
Value: 8.0
Status: CRITICAL
Recommendation: Soil pH is outside the acceptable range. Conduct a soil test and consult an agronomist before applying corrective inputs.

Parameter: nitrogen
Value: 20
Status: CRITICAL


In [25]:
sugarcane_recommendations = generate_crop_recommendations(
    sugarcane_result
)

display_recommendations(
    sugarcane_recommendations
)

Crop: SUGARCANE
Overall Condition: OPTIMAL

Parameter: soil_moisture
Value: 70
Status: OPTIMAL
Recommendation: Soil moisture is within the optimal range. No immediate irrigation action is required.

Parameter: temperature
Value: 29
Status: OPTIMAL
Recommendation: Temperature is within the suitable range for the selected crop.

Parameter: humidity
Value: 70
Status: OPTIMAL
Recommendation: Humidity is within the suitable range for the selected crop.

Parameter: rainfall
Value: 220
Status: OPTIMAL
Recommendation: Rainfall conditions are within the expected range.

Parameter: ph
Value: 6.8
Status: OPTIMAL
Recommendation: Soil pH is within the suitable range. No immediate pH correction is indicated.

Parameter: nitrogen
Value: 110
Status: OPTIMAL
Recommendation: Nitrogen level is within the target range. No immediate nitrogen intervention is indicated.

Parameter: phosphorus
Value: 50
Status: OPTIMAL
Recommendation: Phosphorus level is within the target range. No immediate phosphorus interv

In [26]:
rice_mixed_sensor_data = {

    "soil_moisture": 35,

    "temperature": 27,

    "humidity": 72,

    "rainfall": 220,

    "ph": 6.5,

    "nitrogen": 80,

    "phosphorus": 45,

    "potassium": 60
}

In [27]:
rice_mixed_result = assess_crop_condition(
    crop="rice",
    sensor_data=rice_mixed_sensor_data
)

display_assessment(
    rice_mixed_result
)

Crop: RICE
Overall Condition: CRITICAL
soil_moisture        | Value:       35 | Status: CRITICAL
temperature          | Value:       27 | Status: OPTIMAL
humidity             | Value:       72 | Status: OPTIMAL
rainfall             | Value:      220 | Status: OPTIMAL
ph                   | Value:      6.5 | Status: OPTIMAL
nitrogen             | Value:       80 | Status: OPTIMAL
phosphorus           | Value:       45 | Status: OPTIMAL
potassium            | Value:       60 | Status: OPTIMAL
Summary
Optimal: 7
Critical: 1


In [28]:
rice_mixed_recommendations = generate_crop_recommendations(
    rice_mixed_result
)

display_recommendations(
    rice_mixed_recommendations
)

Crop: RICE
Overall Condition: CRITICAL

Parameter: soil_moisture
Value: 35
Status: CRITICAL
Recommendation: Soil moisture is outside the acceptable range. Immediate field inspection is recommended. Evaluate irrigation requirements before taking action.

Parameter: temperature
Value: 27
Status: OPTIMAL
Recommendation: Temperature is within the suitable range for the selected crop.

Parameter: humidity
Value: 72
Status: OPTIMAL
Recommendation: Humidity is within the suitable range for the selected crop.

Parameter: rainfall
Value: 220
Status: OPTIMAL
Recommendation: Rainfall conditions are within the expected range.

Parameter: ph
Value: 6.5
Status: OPTIMAL
Recommendation: Soil pH is within the suitable range. No immediate pH correction is indicated.

Parameter: nitrogen
Value: 80
Status: OPTIMAL
Recommendation: Nitrogen level is within the target range. No immediate nitrogen intervention is indicated.

Parameter: phosphorus
Value: 45
Status: OPTIMAL
Recommendation: Phosphorus level is w

In [29]:
def run_crop_condition_assessment(
    selected_crop,
    sensor_data
):

    print("\n")
    print("SMART AGRICULTURE AI PLATFORM")
    print("=" * 75)

    print(
        f"Selected Crop: "
        f"{selected_crop.upper()}"
    )

    print("=" * 75)

    # Step 1: Assess crop condition

    assessment = assess_crop_condition(
        crop=selected_crop,
        sensor_data=sensor_data
    )

    # Step 2: Generate recommendations

    recommendations = generate_crop_recommendations(
        assessment
    )

    # Step 3: Display results

    display_recommendations(
        recommendations
    )

    return recommendations

In [30]:
final_rice_result = run_crop_condition_assessment(
    selected_crop="rice",
    sensor_data=rice_mixed_sensor_data
)



SMART AGRICULTURE AI PLATFORM
Selected Crop: RICE
Crop: RICE
Overall Condition: CRITICAL

Parameter: soil_moisture
Value: 35
Status: CRITICAL
Recommendation: Soil moisture is outside the acceptable range. Immediate field inspection is recommended. Evaluate irrigation requirements before taking action.

Parameter: temperature
Value: 27
Status: OPTIMAL
Recommendation: Temperature is within the suitable range for the selected crop.

Parameter: humidity
Value: 72
Status: OPTIMAL
Recommendation: Humidity is within the suitable range for the selected crop.

Parameter: rainfall
Value: 220
Status: OPTIMAL
Recommendation: Rainfall conditions are within the expected range.

Parameter: ph
Value: 6.5
Status: OPTIMAL
Recommendation: Soil pH is within the suitable range. No immediate pH correction is indicated.

Parameter: nitrogen
Value: 80
Status: OPTIMAL
Recommendation: Nitrogen level is within the target range. No immediate nitrogen intervention is indicated.

Parameter: phosphorus
Value: 45
S

In [31]:
final_sugarcane_result = run_crop_condition_assessment(
    selected_crop="sugarcane",
    sensor_data=sugarcane_sensor_data
)



SMART AGRICULTURE AI PLATFORM
Selected Crop: SUGARCANE
Crop: SUGARCANE
Overall Condition: OPTIMAL

Parameter: soil_moisture
Value: 70
Status: OPTIMAL
Recommendation: Soil moisture is within the optimal range. No immediate irrigation action is required.

Parameter: temperature
Value: 29
Status: OPTIMAL
Recommendation: Temperature is within the suitable range for the selected crop.

Parameter: humidity
Value: 70
Status: OPTIMAL
Recommendation: Humidity is within the suitable range for the selected crop.

Parameter: rainfall
Value: 220
Status: OPTIMAL
Recommendation: Rainfall conditions are within the expected range.

Parameter: ph
Value: 6.8
Status: OPTIMAL
Recommendation: Soil pH is within the suitable range. No immediate pH correction is indicated.

Parameter: nitrogen
Value: 110
Status: OPTIMAL
Recommendation: Nitrogen level is within the target range. No immediate nitrogen intervention is indicated.

Parameter: phosphorus
Value: 50
Status: OPTIMAL
Recommendation: Phosphorus level i